# MLP DecompositionDecompose MLP layers into interpretable components — understanding what individual neurons compute, how the MLP transforms features, and where factual knowledge is stored.MLPs act as key-value memories (Geva et al., 2021): W_in columns are "keys" that detect input patterns, and W_out rows are "values" that write specific features to the residual stream.

## Why This Matters

MLP decomposition breaks down the MLP's computation into individual neuron contributions, feature directions, and knowledge storage patterns. Understanding how nonlinearities (GELU/ReLU) interact with the key-value structure reveals the mechanisms of MLP computation at a fine-grained level.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores

In [ ]:
import jaximport jax.numpy as jnpimport numpy as npfrom irtk import HookedTransformerConfig, HookedTransformerfrom irtk.mlp_decomposition import (    neuron_contribution_decompose,    mlp_feature_directions,    mlp_input_output_alignment,    mlp_knowledge_storage,    mlp_nonlinearity_analysis,)cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)model = HookedTransformer(cfg)key = jax.random.PRNGKey(42)leaves, treedef = jax.tree.flatten(model)new_leaves = []for leaf in leaves:    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:        key, sk = jax.random.split(key)        new_leaves.append(jax.random.normal(sk, leaf.shape, dtype=leaf.dtype) * 0.1)    else:        new_leaves.append(leaf)model = jax.tree.unflatten(treedef, new_leaves)tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])

## Neuron Contribution DecompositionDecompose MLP output into per-neuron contributions. Each neuron contributes `activation * W_out_row` to the residual stream.

In [ ]:
result = neuron_contribution_decompose(model, tokens, layer=0, top_k=5)print(f"Top {len(result['top_neurons'])} neurons by contribution:")for idx, norm in result['top_neurons']:    print(f"  Neuron {idx}: contribution norm = {norm:.6f}")print(f"\nTotal MLP output norm: {result['total_contribution']:.4f}")print(f"Top-5 explain {result['top_k_fraction']*100:.1f}% of total output")

## MLP Feature DirectionsExtract the principal input (key) and output (value) directions from the MLP weight matrices. These reveal the MLP's "vocabulary" of features.

In [ ]:
result = mlp_feature_directions(model, layer=0, top_k=5)print(f"Effective rank of MLP: {result['effective_rank']:.2f}")print(f"Top singular values: {[f'{s:.4f}' for s in result['singular_values']]}")print(f"Input direction shapes: {result['input_directions'].shape}")print(f"Output direction shapes: {result['output_directions'].shape}")

## Input-Output AlignmentAre MLP neurons amplifying existing features (high alignment) or creating new ones (low alignment)?

In [ ]:
result = mlp_input_output_alignment(model, layer=0)print(f"Mean |alignment|: {result['mean_alignment']:.4f}")print(f"Amplifying neurons (|align| > 0.5): {result['amplifying_neurons']}")print(f"Transforming neurons (|align| < 0.1): {result['transforming_neurons']}")print(f"Alignment range: [{result['per_neuron_alignment'].min():.4f}, {result['per_neuron_alignment'].max():.4f}]")

## Knowledge StorageWhich neurons promote a specific target token? Following Geva et al. (2022), each neuron's output direction is projected through the unembedding matrix to measure its effect on target logits.

In [ ]:
result = mlp_knowledge_storage(model, tokens, layer=0, target_token=5, top_k=5)print(f"Top knowledge neurons for token 5:")for idx, effect in result['knowledge_neurons']:    print(f"  Neuron {idx}: logit effect = {effect:+.6f}")print(f"\nTotal promotion: {result['total_promotion']:.6f}")print(f"Knowledge concentration (top-5): {result['knowledge_concentration']*100:.1f}%")

## Nonlinearity AnalysisAnalyze the activation function's gating behavior — which neurons are active, how sharp the gating is, and whether there are "dead" neurons.

In [ ]:
result = mlp_nonlinearity_analysis(model, tokens, layer=0)print(f"Active fraction: {result['active_fraction']*100:.1f}%")print(f"Dead neurons: {result['dead_neurons']}")print(f"Gating sharpness: {result['gating_sharpness']:.4f}")print(f"Pre-activation stats:")stats = result['pre_activation_stats']print(f"  Mean: {stats['mean']:.4f}, Std: {stats['std']:.4f}")print(f"  Range: [{stats['min']:.4f}, {stats['max']:.4f}]")